In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
TRAIN_CSV = "../data/kaggle/fashion-mnist_train.csv"
TEST_CSV = "../data/kaggle/fashion-mnist_test.csv"

BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
VALID_RATIO = 0.2 
RANDOM_SEED = 42

CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
class FashionMNISTCSVDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(csv_file)

        self.labels = df.iloc[:, 0].to_numpy(dtype=np.int64)
        self.images = df.iloc[:, 1:].to_numpy(dtype=np.float32) / 255.0  
        self.images = self.images.reshape(-1, 1, 28, 28) 

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = torch.from_numpy(self.images[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

In [4]:
test_dataset = FashionMNISTCSVDataset(TEST_CSV)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

In [5]:
class FashionCNNFlexible(nn.Module):
    def __init__(
        self,
        conv_channels=[32, 64],
        kernel_size=3,
        pool_type="max",
        activation="relu",
        dropout=0.3,
        use_batchnorm=False,
        num_classes=10
    ):
        super().__init__()

        layers = []
        in_channels = 1
        current_size = 28  
        
        def get_activation():
            if activation.lower() == "relu":
                return nn.ReLU()
            elif activation.lower() == "gelu":
                return nn.GELU()
            else:
                raise ValueError(f"Unsupported activation: {activation}")

        def get_pool():
            if pool_type.lower() == "max":
                return nn.MaxPool2d(kernel_size=2, stride=2)
            elif pool_type.lower() == "avg":
                return nn.AvgPool2d(kernel_size=2, stride=2)
            else:
                raise ValueError(f"Unsupported pool type: {pool_type}")

        padding = kernel_size // 2  # keep spatial size same after conv

        for out_channels in conv_channels:
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding))
            # (current_size + 2*padding - kernel_size)/stride + 1 = input, so padding = kernel_size // 2 keeps the size same
            
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_channels))
            
            layers.append(get_activation())
            layers.append(get_pool())

            in_channels = out_channels
            current_size = current_size // 2  # after 2x2 pooling

        self.features = nn.Sequential(*layers)

        flattened_dim = conv_channels[-1] * current_size * current_size

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_dim, 128),
            get_activation(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [6]:
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad(): # tells PyTorch not to calculate gradients, which saves memory and computations during evaluation
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy()) # This saves all predictions and actual labels across all batches
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

In [7]:
import pandas as pd
import numpy as np
import torch

# 1. Define all model configurations (from your previous work)
model_configs = {
    "V1_Baseline": {"conv_channels": [32, 64], "kernel_size": 3, "pool_type": "max", "activation": "relu", "dropout": 0.3, "use_batchnorm": False},
    "V2_BatchNorm": {"conv_channels": [32, 64], "kernel_size": 3, "pool_type": "max", "activation": "relu", "dropout": 0.3, "use_batchnorm": True},
    "V3_LargerKernel": {"conv_channels": [32, 64], "kernel_size": 5, "pool_type": "max", "activation": "relu", "dropout": 0.3, "use_batchnorm": False},
    "V4_AvgPool": {"conv_channels": [32, 64], "kernel_size": 3, "pool_type": "avg", "activation": "relu", "dropout": 0.3, "use_batchnorm": False},
    "V5_GELU": {"conv_channels": [32, 64], "kernel_size": 3, "pool_type": "max", "activation": "gelu", "dropout": 0.3, "use_batchnorm": False},
    "V6_NoDropout": {"conv_channels": [32, 64], "kernel_size": 3, "pool_type": "max", "activation": "relu", "dropout": 0.0, "use_batchnorm": False},
    "V7_Deeper": {"conv_channels": [16, 32, 64], "kernel_size": 3, "pool_type": "max", "activation": "relu", "dropout": 0.3, "use_batchnorm": True}
}

num_folds = 5
all_fold_details = [] # To store every fold's data
summary_results = []  # To store averages per model

criterion = nn.CrossEntropyLoss()

for model_name, config in model_configs.items():
    print(f"\nEvaluating Architecture: {model_name}")
    print("-" * 40)
    
    fold_losses = []
    fold_accuracies = []
    
    for fold in range(1, num_folds + 1):
        # Define model with specific config and load fold weights
        model = FashionCNNFlexible(**config).to(device)
        model_path = f"saved_models_kfold/{model_name}_fold{fold}_best_loss.pth"
        
        if os.path.exists(model_path):
            model.load_state_dict(torch.load(model_path, map_location=device))
            test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)
            
            fold_losses.append(test_loss)
            fold_accuracies.append(test_acc)
            
            # Save detail for this specific fold
            all_fold_details.append({
                "Model": model_name,
                "Fold": fold,
                "Test_Loss": test_loss,
                "Test_Accuracy": test_acc
            })
            print(f"Fold {fold} - Loss: {test_loss:.4f}, Acc: {test_acc:.4f}")
        else:
            print(f"Skipping: {model_path} not found.")

    # Calculate averages for this model
    if fold_losses:
        avg_loss = np.mean(fold_losses)
        avg_acc = np.mean(fold_accuracies)
        summary_results.append({
            "Model": model_name,
            "Avg_Test_Loss": avg_loss,
            "Avg_Test_Accuracy": avg_acc,
            "Std_Accuracy": np.std(fold_accuracies) # Useful to see stability
        })
        print(f">> Average Acc: {avg_acc:.4f}")

# 2. Save to CSVs
# Full details (35 rows: 7 models * 5 folds)
df_details = pd.DataFrame(all_fold_details)
df_details.to_csv("results/all_models_all_folds_details.csv", index=False)

# Summary results (7 rows: 1 per model)
df_summary = pd.DataFrame(summary_results)
df_summary.to_csv("results/model_final_averages.csv", index=False)

print("\nEvaluation complete. Results saved to 'results/' folder.")


Evaluating Architecture: V1_Baseline
----------------------------------------


/Users/jiacheng/Documents/NUS_Course/CS3244/group_project/cnn_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Fold 1 - Loss: 0.2134, Acc: 0.9211
Fold 2 - Loss: 0.2084, Acc: 0.9277
Fold 3 - Loss: 0.2139, Acc: 0.9248
Fold 4 - Loss: 0.2158, Acc: 0.9254
Fold 5 - Loss: 0.2054, Acc: 0.9260
>> Average Acc: 0.9250

Evaluating Architecture: V2_BatchNorm
----------------------------------------
Fold 1 - Loss: 0.2092, Acc: 0.9276
Fold 2 - Loss: 0.2055, Acc: 0.9258
Fold 3 - Loss: 0.2148, Acc: 0.9228
Fold 4 - Loss: 0.1994, Acc: 0.9297
Fold 5 - Loss: 0.2226, Acc: 0.9209
>> Average Acc: 0.9254

Evaluating Architecture: V3_LargerKernel
----------------------------------------
Fold 1 - Loss: 0.2095, Acc: 0.9219
Fold 2 - Loss: 0.2161, Acc: 0.9201
Fold 3 - Loss: 0.2096, Acc: 0.9213
Fold 4 - Loss: 0.2230, Acc: 0.9198
Fold 5 - Loss: 0.2199, Acc: 0.9218
>> Average Acc: 0.9210

Evaluating Architecture: V4_AvgPool
----------------------------------------
Fold 1 - Loss: 0.1971, Acc: 0.9298
Fold 2 - Loss: 0.2083, Acc: 0.9248
Fold 3 - Loss: 0.2083, Acc: 0.9252
Fold 4 - Loss: 0.2085, Acc: 0.9238
Fold 5 - Loss: 0.2021, Ac